# Lesson 10 (Going Further): Mixed Models for Binary Data — GLMMs
### Student Laboratory Workbook
*Course: Bayesian Analysis of Empirical Data (2026)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/10_mixed_models_logistic.ipynb)
*Russian version: [10_mixed_models_logistic_ru.ipynb](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/10_mixed_models_logistic_ru.ipynb)*

Lesson 10 taught one binary model per observation. Real binary data, however, is often *longitudinal*: the same patient answers at several visits, the same citizen is tracked across waves. This companion to the *Mixed Models vs. Hierarchical Bayes* page adds the random-effects machinery to the logistic link, on two datasets: the classic **depression drug trial** from Agresti's *Categorical Data Analysis* (340 patients, three follow-ups) and the **ANES 2016** vote data. We climb the same ladder as in Lesson 9's linear workbook — flat model, naive flat model, classical GLMM, Bayesian hierarchical GLMM — and let PSIS-LOO arbitrate.


In [ ]:
import sys, os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

try:
    import arviz as az
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arviz>=0.14.0"])
    import arviz as az

try:
    import bambi as bmb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bambi>=0.13.0"])
    import bambi as bmb

RANDOM_SEED = 2026
rng = np.random.default_rng(RANDOM_SEED)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

def fetch(fname, url):
    # Canonical copy in the book repository; auto-download in Colab or when missing.
    path = os.path.join("data", fname)
    os.makedirs("data", exist_ok=True)
    if not os.path.exists(path):
        import urllib.request
        urllib.request.urlretrieve(url, path)
    return path

BASE = "https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/"
DEPR_PATH = fetch("Depression.dat", BASE + "Depression.dat")
ANES_PATH = fetch("anes_clinton.csv", BASE + "anes_clinton.csv")

print("Environment ready: statsmodels", sm.__version__, "| Bambi", bmb.__version__, "| ArviZ", az.__version__)


## 1. A Longitudinal Binary Outcome: the Depression Drug Trial

### The Idea
The study (distributed as Table 9.1 in Agresti's *Categorical Data Analysis* materials) is a longitudinal clinical trial comparing a **new drug** with a **standard** drug for subjects suffering mental depression. Subjects were classified by initial severity — *mild* or *severe* — randomly assigned to one of the two drugs, and their status was classified as *normal* (1) or *abnormal* (0) after 1, 2 and 4 weeks of treatment.

A modeling detail that matters: when the time metric reflects **cumulative dosage**, a logit model often behaves approximately linearly in the **logarithm of time**. Weeks 1, 2, 4 give the tidy scores $t = \log_2(\text{week}) = 0, 1, 2$. Agresti reports similar substantive results with raw week numbers; the log scores are the theoretically motivated default.

### Example


In [ ]:
data_depr = pd.read_csv(DEPR_PATH, sep=r"\s+")
print(data_depr.head())
print("\nShape:", data_depr.shape,
      "| cases:", data_depr["case"].nunique(),
      "| rows per case:", sorted(data_depr.groupby("case").size().unique()))
print("severity: 0 = mild, 1 = severe | drug: 0 = standard, 1 = new | time: log2(week) in {0,1,2}")

# Descriptive proportion of 'normal' responses by cell
agg = data_depr.groupby(["severity", "drug", "time"])["outcome"].mean().reset_index()
print("\nShare of normal outcomes by cell:")
print(agg.pivot(index=["severity", "drug"], columns="time", values="outcome").round(3))

sns.catplot(x="time", y="outcome", hue="drug", col="severity", data=agg, kind="bar",
            palette="rocket", legend_out=False, height=3.6, aspect=1.1)
plt.suptitle("Proportion normal by week, drug and baseline severity", y=1.05)
plt.show()


### What to Watch For
Read the four panels as a $2 \times 2$ factorial story:
- Both arms start near their baseline severity level (mild $\approx 0{.}5$, severe $\approx 0{.}2$).
- Under the **standard** drug improvement is slow and modest in both severity groups.
- Under the **new** drug the curves take off: by week 4 (t=2) 97% of initially mild and 83% of initially severe patients are classified normal.
- The gap between drugs *grows with time* — a first hint at a **drug × time interaction** on the logit scale, which the flat models below will formalize.

And the structural point: the same `case` id appears three times. These 1020 rows are not 1020 independent people; they are 340 people measured thrice.


## 2. Flat Logistic Models: the Naive Baselines

### The Idea
Pretend all rows are independent and fit ordinary logistic regressions — first with `drug` only, then with the full fixed structure `drug + time + severity + drug:time`. This is exactly the "complete pooling" extreme from Lesson 9, now on a binary outcome. The comparison between the two fits already tells the substantive story; the residual plots show why "residuals" for binary data must be read with care.

### Example


In [ ]:
basic1 = smf.logit(formula="outcome ~ drug", data=data_depr).fit(disp=0)
print(basic1.summary())
print("\nOdds ratio for drug:", round(float(np.exp(basic1.params["drug"])), 2))


In [ ]:
basic2 = smf.logit(formula="outcome ~ drug + time + severity + drug:time", data=data_depr).fit(disp=0)
print(basic2.summary())

# Binned residuals: for binary y, raw residuals always form two bands — bin them first.
def binned_residuals(res, data, x="fitted", n_bins=10, ax=None):
    pred = res.fittedvalues if x == "fitted" else data[x]
    df = pd.DataFrame({"x": pred, "r": res.resid_generalized})
    if df["x"].nunique() <= n_bins:
        df["bin"] = df["x"]          # discrete designs (e.g. drug-only): bin by fitted value
    else:
        df["bin"] = pd.qcut(df["x"], n_bins)
    binned = df.groupby("bin", observed=True).mean(numeric_only=True)
    ax.plot(binned["x"], binned["r"], "o-", color="C3")
    ax.axhline(0, color="gray", lw=0.8)
    return binned

fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].scatter(basic1.fittedvalues, basic1.resid_generalized, s=8, alpha=0.5, color="C0")
axs[0].set_title("drug-only: raw residuals vs fitted")
binned_residuals(basic1, data_depr, ax=axs[1])
axs[1].set_title("drug-only: binned mean residuals")
for ax in axs: ax.set_xlabel("fitted probability")
plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].scatter(basic2.fittedvalues, basic2.resid_generalized, s=8, alpha=0.5, color="C0")
axs[0].set_title("full flat model: raw residuals vs fitted")
binned_residuals(basic2, data_depr, ax=axs[1])
axs[1].set_title("full flat model: binned mean residuals")
for ax in axs: ax.set_xlabel("fitted probability")
plt.tight_layout(); plt.show()


### What to Watch For
1. In the drug-only model the new drug looks uniformly better ($\widehat{\operatorname{OR}} \approx 2$). In the full model the **main effect of `drug` collapses to $\approx -0.06$** while `drug:time` explodes to $\approx +1.02$: the drugs are similar at baseline and diverge with cumulative dosage. A main effect without its interaction is a *weighted average over time* — a meaningless summary when the interaction is real.
2. The raw-residual panels are two bands by construction ($y - p$ given $y \in \{0,1\}$); they carry no diagnostic message. The *binned* mean residuals are the useful version: for the drug-only model they curve systematically — structure is missing. (Binned residual plots are the binary-outcome analogue of residual-vs-fitted checks, following Gelman & Hill.)
3. Every number here is computed under the fiction of independent rows, so all standard errors are too small — the price of ignoring `case`.


## 3. The Classical GLMM Attempt: `BinomialBayesMixedGLM`

### The Idea
Add a random intercept per patient: `outcome ~ drug + time + severity + drug:time + (1 | case)`. Exact likelihood for a logistic GLMM requires numerical integration over 340 patient effects, which classical software sidesteps with approximations. `statsmodels` offers a *variational Bayes* approximation (`fit_vb`): it returns approximate posterior means and standard deviations for the fixed effects, but **no posterior samples, no convergence diagnostics, no full posterior for the variance component** — a Gaussian approximation around a mode, not the real thing.

### Example


In [ ]:
random_effects_spec = {"a": "0 + C(case)"}
mixed_vb = BinomialBayesMixedGLM.from_formula(
    "outcome ~ drug + time + severity + drug:time", random_effects_spec, data_depr)
result_vb = mixed_vb.fit_vb()
print(result_vb.summary())

fe = pd.DataFrame({"VB mean": result_vb.fe_mean,
                   "VB sd": result_vb.fe_sd,
                   "flat logit": basic2.params.values},
                  index=["Intercept", "drug", "time", "severity", "drug:time"])
print("\nFixed effects: VB mixed GLMM vs flat logit MLE")
print(fe.round(3))

re_vb = result_vb.random_effects()
print("\nFirst patient effects (VB):")
print(re_vb.head().round(3))
print("Estimated SD of case effects (VB):", round(float(result_vb.vc_mean[0]), 3))


### What to Watch For
The VB point estimates land close to the flat MLE — the fixed-effect story is stable. What the table cannot give you:
- a posterior distribution for the case-level standard deviation (only one number `vc_mean`);
- a predictive distribution for a *new* patient;
- any way to check whether the variational approximation is good (the ELBO improved, but by how much is it off?).

The legacy notebook said it bluntly: with point estimates alone "it is difficult to assess quality and make quantitative claims — by how much, which effect size, what uncertainty?" The Bayesian hierarchical model below answers exactly those questions.


## 4. The Bayesian Hierarchical GLMM in Bambi + PSIS-LOO

### The Idea
The same formula, now with a full posterior: every patient gets an intercept $b_{\text{case}}$ drawn from $\operatorname{Normal}(0, \sigma_{\text{case}})$, and $\sigma_{\text{case}}$ itself is a parameter with a posterior distribution. We fit three nested models and let PSIS-LOO (the course criterion; the legacy notebook printed WAIC, which ArviZ 1.x no longer ships) rank them by out-of-sample predictive accuracy:
1. `drug` only — the naive pooling baseline;
2. `drug + time + severity + drug:time` — the full flat model;
3. the same plus `(1 | case)` — the hierarchical GLMM.

### Example


In [ ]:
FIXED = "drug + time + severity + drug:time"

model1 = bmb.Model("outcome ~ drug", data_depr, family="bernoulli")
idata1 = model1.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                    idata_kwargs={"log_likelihood": True})
print(az.summary(idata1, round_to=2))


In [ ]:
model2 = bmb.Model(f"outcome ~ {FIXED}", data_depr, family="bernoulli")
idata2 = model2.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                    idata_kwargs={"log_likelihood": True})
print(az.summary(idata2, round_to=2))
print("\nPSIS-LOO (ELPD):", az.loo(idata2).elpd.round(1))


In [ ]:
model3 = bmb.Model(f"outcome ~ {FIXED} + (1 | case)", data_depr, family="bernoulli")
print(model3)
idata3 = model3.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                    target_accept=0.95, idata_kwargs={"log_likelihood": True})
print(az.summary(idata3, var_names=["Intercept", "drug", "time", "severity",
                                    "drug:time", "1|case_sigma"], round_to=2))
print("\nPSIS-LOO (ELPD):", az.loo(idata3).elpd.round(1))


In [ ]:
cmp_df = az.compare({"M1: drug only (flat)": idata1,
                        "M2: full fixed (flat)": idata2,
                        "M3: + (1|case) GLMM": idata3})
print(cmp_df.round(1))
az.plot_compare(cmp_df)
plt.show()


### What to Watch For
1. **Trust checks first**: `r_hat` $\le 1.01$ and healthy ESS in the summaries above. Even at `target_accept=0.95` the GLMM reports a handful of divergences: with only three binary observations per patient, $\sigma_{\text{case}}$ is weakly identified, its posterior piles up near zero, and the sampler feels the funnel geometry of Lesson 6 — the same boundary territory as the "singular fit" of `lme4` on the book page.
2. **The LOO verdict is a tie**: the full flat model and the GLMM differ by about one ELPD (dse $< 1$). With three observations per patient the data cannot tell "no patient heterogeneity" from "small patient heterogeneity": partial pooling degenerates toward complete pooling, and out-of-sample prediction does not care. Contrast this with `sleepstudy` in the Lesson 9 workbook, where ten continuous observations per subject let the hierarchical structure win decisively. **Hierarchy earns its keep when clusters are informative — not merely because rows repeat.**
3. The posterior of `1|case_sigma` (mean $\approx 0.3$, mass concentrated near zero) is precisely the answer the VB table could only hint at: the full uncertainty statement about between-patient heterogeneity that the classical fit could not make — and here it says the heterogeneity is small.
4. Pareto-$k$ warnings above $0.7$ are common for hierarchical models (each row leans on its patient's effect); treat them as a prompt to consider leave-one-*patient*-out CV rather than a verdict of failure.

The next cell reports the model where it matters — on the **observable probability scale**, with named baselines (the course protocol: report contrasts at stated baseline values, never bare log-odds):


In [ ]:
def cell_probability(idata, severity, drug, t, var_names):
    # Posterior of P(normal) for a TYPICAL patient (case effect = 0)
    post = idata.posterior.to_dataset()
    eta = (post["Intercept"] + drug * post["drug"] + t * post["time"]
           + severity * post["severity"] + drug * t * post["drug:time"])
    return 1 / (1 + np.exp(-eta)).values.flatten()

rows = []
for sev, sev_name in [(0, "mild"), (1, "severe")]:
    for dr, dr_name in [(0, "standard"), (1, "new")]:
        for t, week in [(0, 1), (1, 2), (2, 4)]:
            p = cell_probability(idata3, sev, dr, t, None)
            lo, hi = az.hdi(p, prob=0.94)
            rows.append({"severity": sev_name, "drug": dr_name, "week": week,
                         "P(normal) mean": p.mean().round(3),
                         "94% HDI": f"[{lo:.2f}, {hi:.2f}]"})
report = pd.DataFrame(rows)
print("M3 (hierarchical GLMM), typical patient, posterior P(normal):")
print(report.to_string(index=False))

# Probability-scale drug effect at week 4 for each severity
for sev, sev_name in [(0, "mild"), (1, "severe")]:
    p_new = cell_probability(idata3, sev, 1, 2, None)
    p_std = cell_probability(idata3, sev, 0, 2, None)
    delta = p_new - p_std
    lo, hi = az.hdi(delta, prob=0.94)
    print(f"week 4, {sev_name}: P(new) - P(standard) = {delta.mean():+.3f}, 94% HDI [{lo:+.2f}, {hi:+.2f}]")


## 5. Cross-Sectional Interactions: ANES 2016 Vote

### The Idea
Not all binary data is longitudinal, but the same machinery handles *heterogeneous groups* in cross-sections too. The ANES 2016 example (from the Bambi documentation) models the Clinton/Trump vote with **party-specific age slopes**. One coding subtlety: the formula `vote['clinton'] ~ party_id + party_id:age` expands the interaction into an age slope **for every party** (three slopes: democrat, independent, republican), so the model is full-rank — the age main effect is absorbed into the per-party slopes rather than omitted. `vote['clinton']` tells Bambi which level of the response is the modeled "success".

### Example


In [ ]:
clinton_data = pd.read_csv(ANES_PATH)
print(clinton_data.head())
print("\nRows:", len(clinton_data), "| votes:", clinton_data["vote"].value_counts().to_dict(),
      "| parties:", sorted(clinton_data["party_id"].unique()))

clinton_model = bmb.Model("vote['clinton'] ~ party_id + party_id:age", clinton_data,
                          family="bernoulli")
print(clinton_model)
idata_anes = clinton_model.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                               target_accept=0.85, idata_kwargs={"log_likelihood": True})
print(az.summary(idata_anes, round_to=2))


In [ ]:
# Report the age effects on the observable scale: probability change per decade of age,
# evaluated at each party's own baseline (mean age within party).
post = idata_anes.posterior.to_dataset()
party_levels = ["democrat", "independent", "republican"]
base_age = {p: clinton_data.loc[clinton_data["party_id"] == p, "age"].mean()
            for p in party_levels}
slopes = {p: post["party_id:age"].sel(**{"party_id:age_dim": p}).values.flatten()
          for p in party_levels}

fig, ax = plt.subplots(figsize=(7, 3.8))
for i, p in enumerate(party_levels):
    ax.hist(slopes[p], bins=60, alpha=0.55, density=True, label=p, color=f"C{i}")
ax.set_xlabel("posterior log-odds slope per year of age")
ax.set_title("Party-specific age slopes (ANES 2016, Clinton vote)")
ax.legend()
plt.tight_layout(); plt.show()

print("Per-decade effect on P(vote Clinton), at each party's mean age (divide-by-4 rule as a sanity check):")
for p in party_levels:
    s10 = slopes[p] * 10
    eta = post["Intercept"].values.flatten()
    if p != "democrat":   # 'democrat' is the reference level, absorbed into the Intercept
        eta = eta + post["party_id"].sel(**{"party_id_dim": p}).values.flatten()
    p_hat = 1 / (1 + np.exp(-eta))
    delta_prob = s10 * p_hat * (1 - p_hat)   # logit-approximation of the probability change
    lo, hi = az.hdi(delta_prob, prob=0.94)
    print(f"  {p:12s} baseline age {base_age[p]:.0f}: "
          f"DeltaP/decade = {delta_prob.mean():+.3f}, 94% HDI [{lo:+.3f}, {hi:+.3f}]")


### What to Watch For
1. The three posterior slope distributions barely overlap: age *negatively* predicts a Clinton vote among republicans and independents, while among democrats the slope is much closer to zero. An "average age effect" would have hidden the entire story — interactions are how heterogeneity becomes visible.
2. We translated slopes to the probability scale **at each party's own baseline probability** (the course's golden rule from Lesson 10): the same logit slope means different probability changes for democrats ($p \approx 0.9$) and republicans ($p \approx 0.1$).
3. The source notebook used `draws=200, target_accept=0.85`; we raised the draws to 1000 and kept `target_accept=0.85` — always check the r_hat column before quoting the numbers.


## 6. Optional Homework: the Prosocial Chimpanzees, Revisited as a GLMM

The legacy notebook pointed to "starfish and chimpanzees GLM examples" from other course materials. The starfish pointer had no attached dataset; the chimpanzees are already in this book as the Lesson 10 capstone — so the homework makes the connection explicit.

**Tasks:**
1. Load the chimpanzees data from the book repository (`data/chimpanzees.csv`, see the Lesson 10 workbook) and recall the design: `pulled_left`, `prosoc_left`, `condition`, `actor`, `block`.
2. Fit the flat logistic model `pulled_left ~ prosoc_left * condition` ignoring actors and blocks.
3. Fit the GLMM `pulled_left ~ prosoc_left * condition + (1 | actor) + (1 | block)` in Bambi.
4. Compare the two with PSIS-LOO. Report the population-level effect of the prosocial option on P(pull left) with its uncertainty, and one actor-level posterior contrast of your choice.
5. One sentence on what `(1 | block)` buys you — and check whether the LOO table supports including it.


In [ ]:
# Homework data starter (no solutions below — this one is yours to build)
CHIMP_PATH = fetch("chimpanzees.csv", BASE + "chimpanzees.csv")
chimp = pd.read_csv(CHIMP_PATH, sep=";")
print(chimp.head())
print("actors:", chimp["actor"].nunique(), "| blocks:", chimp["block"].nunique(),
      "| trials:", len(chimp))


## Summary

- Longitudinal binary data has the same clustered structure as longitudinal continuous data; the fix is the same — random effects — wrapped around a different link function.
- The flat logistic ladder already carries the substantive story (main effect of the drug collapses once the interaction with time enters), but its standard errors are fiction.
- `statsmodels`' variational GLMM gives point estimates and approximate SDs; the Bayesian hierarchical GLMM gives the full posterior — including the between-patient SD — and PSIS-LOO arbitration between the flat and hierarchical structures. In this trial the arbitration called it a tie (three binary rows per patient carry too little cluster information); in `sleepstudy` hierarchy won decisively.
- Reporting belongs on the probability scale at named baselines: "at week 4, initially mild patients: +X probability points under the new drug, 94% HDI [a, b]".
- Reference literature: Agresti, *Categorical Data Analysis*, the depression clinical trial example; Gelman & Hill, *Data Analysis Using Regression and Multilevel/Hierarchical Models*, binned residuals and varying-intercept logistic models; the Bambi documentation for the ANES case.

**Back to the book:** the Lesson 10 capstone (Prosocial Chimpanzees) builds a hierarchical logit by hand; the *Mixed Models vs. Hierarchical Bayes* page condenses this ladder; Lesson 11 adds the full diagnostic arsenal.
